# Analisi emotiva di un singolo commento con ELIta

Questo notebook come il **metodo finale** (_lessico ELIta ibrido (α=0.5) + corpus_mean normalisation (Formula 3.5 ItEm)_) assegna un'emozione a un commento del corpus `r/Italia: notizie, film, sport`.

## Setup e caricamento dati

In [4]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from Fase3.support import (BASIC_EMOTIONS, POS_FILTER, load_corpus, compute_mu_e,
                           normalizza, emozione_dominante, plot_radar_single, plot_raw_vs_norm_bars, load_elita_matrix, )

CORPUS_CSV   = Path('corpus_Italia_multi.csv')
TOKENS_CSV   = Path('tokens_Italia_multi.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')

print('Configurazione caricata.')

Configurazione caricata.


In [5]:
df_corpus, df_tokens = load_corpus(CORPUS_CSV, TOKENS_CSV)
df_elita_final = load_elita_matrix(ALPHA_05_CSV)
mu_e = compute_mu_e(df_corpus, df_tokens, df_elita_final)

print('Corpus:', len(df_corpus))
print('Token:', len(df_tokens))
print('Post:', (df_corpus['type'] == 'post').sum())
print('Commenti:', (df_corpus['type'] == 'comment').sum())

Corpus: 4311
Token: 146186
Post: 284
Commenti: 4027


In [6]:
print(df_tokens['pos'].value_counts().to_string())

pos
NOUN     29487
ADP      20016
VERB     19639
DET      15450
ADV      13240
PRON     10548
ADJ       9468
AUX       8566
PROPN     6978
CCONJ     5695
SCONJ     3819
NUM       2028
X          568
INTJ       307
EMOJI      177
PUNCT       95
SYM         87
PART        18


## Selezione del documento

In [7]:
DOC_INDEX = 1616       # valore tra (0-5001)
DOC_ID    = None       # oppure specifica un ID, es. 'kfr4gvl' (commento) o '1idmjsb' (post)
DOC_TYPE  = 'comment'  # 'comment', 'post', oppure None per tutti

df_sel = df_corpus[df_corpus['type'] == DOC_TYPE] if DOC_TYPE else df_corpus # Filtra per tipo se richiesto
df_sel = df_sel.reset_index(drop=True)

if DOC_ID:
    row = df_sel[df_sel['doc_id'] == DOC_ID].iloc[0]
else:
    row = df_sel.iloc[DOC_INDEX]

CID  = row['doc_id']
TEXT = row['text']

print(f'ID          : {CID}')
print(f'Tipo        : {row["type"]}')
print(f'Autore      : {row["author"]}')
print(f'Score Reddit: {row["score"]}')
print()
print('Testo:')
print('-' * 100)
print(TEXT)
print('-' * 100)

ID          : lbxx0f1
Tipo        : comment
Autore      : Italia-ModTeam
Score Reddit: 2

Testo:
----------------------------------------------------------------------------------------------------
###Violenza, molestie, bullismo

Sono vietati i contenuti che costituiscono o incitano a violenza, molestie, bullismo, anche contro soggetti esterni a reddit. Questa è una regola globale della [Content Policy di reddit](https://www.redditinc.com/it-it/policies/content-policy); la sua violazione può comportare la sospensione dal sito.

https://www.reddit.com/r/Italia/wiki/rules
----------------------------------------------------------------------------------------------------


## Token e lemmi del commento

In [8]:
df_tok = df_tokens[df_tokens['doc_id'] == CID].copy()
elita_idx_all = set(df_elita_final.index)

df_tok['in_ELIta'] = df_tok['lemma'].isin(elita_idx_all)
df_tok['pos_ok']   = df_tok['pos'].isin(POS_FILTER)
df_tok['usato']    = df_tok['in_ELIta'] & df_tok['pos_ok']

print(f'Token totali nel documento          : {len(df_tok)}')
print(f'Con POS valida (ADJ/NOUN/VERB/AUX/EMOJI): {df_tok["pos_ok"].sum()}')
print(f'Trovati in ELIta                    : {df_tok["in_ELIta"].sum()}')
print(f'Token usati per l\'analisi           : {df_tok["usato"].sum()}')
print()

display(df_tok[['token','lemma','pos','in_ELIta','usato']].reset_index(drop=True))

Token totali nel documento          : 46
Con POS valida (ADJ/NOUN/VERB/AUX/EMOJI): 27
Trovati in ELIta                    : 17
Token usati per l'analisi           : 17



,token,lemma,pos,in_ELIta,usato
0,Violenza,violenza,NOUN,True,True
1,molestie,molestia,NOUN,False,False
2,bullismo,bullismo,NOUN,True,True
3,Sono,essere,AUX,False,False
4,vietati,vietare,VERB,True,True
5,i,il,DET,False,False
6,contenuti,contenuto,NOUN,True,True
7,che,che,PRON,False,False
8,costituiscono,costituire,VERB,True,True
9,o,o,CCONJ,False,False


Anche se ELIta è descritta come composta da aggettivi, nomi e verbi, in realtà contiene anche altri POS (come AUX e EMOJI) che possono contribuire all'analisi emotiva. Per questo motivo, consideriamo utili tutti i token che appartengono a POS validi e sono presenti in ELIta.

## Contributo emotivo per parola

Per ogni lemma usato nell'analisi, mostriamo il vettore emotivo da ELIta α=0.5 (score grezzi).

In [10]:
lemmi_usati = df_tok[df_tok['usato']]['lemma'].tolist()

if not lemmi_usati:
    print('Nessun lemma utile trovato in questo commento.')
else:
    contrib_rows = []
    for lemma in lemmi_usati:
        scores = df_elita_final.loc[lemma, BASIC_EMOTIONS].to_dict()
        dom    = max(scores, key=scores.get)
        scores['lemma']   = lemma
        scores['dom_emo'] = dom
        contrib_rows.append(scores)

    df_contrib = pd.DataFrame(contrib_rows)
    cols_show  = ['lemma'] + BASIC_EMOTIONS + ['dom_emo']

    totals  = df_contrib[BASIC_EMOTIONS].sum()
    dom_raw = totals.idxmax()

    print('Contributi emotivi per lemma (ELIta α=0.5 — score grezzi):')
    display(
        df_contrib[cols_show]
        .style
        .background_gradient(subset=BASIC_EMOTIONS, cmap='YlOrRd', axis=None)
        .format({e: '{:.2f}' for e in BASIC_EMOTIONS})
    )

Contributi emotivi per lemma (ELIta α=0.5 — score grezzi):


,lemma,gioia,aspettativa,rabbia,disgusto,tristezza,sorpresa,paura,fiducia,dom_emo
0,violenza,0.05,0.17,0.90,0.86,0.81,0.18,0.83,0.07,rabbia
1,bullismo,0.05,0.12,0.92,0.85,0.76,0.29,0.76,0.07,rabbia
2,vietare,0.08,0.20,0.79,0.47,0.56,0.34,0.70,0.28,rabbia
3,contenuto,0.64,0.76,0.32,0.30,0.31,0.68,0.35,0.68,aspettativa
4,costituire,0.67,0.71,0.08,0.10,0.20,0.32,0.15,0.72,fiducia
5,violenza,0.05,0.17,0.90,0.86,0.81,0.18,0.83,0.07,rabbia
6,bullismo,0.05,0.12,0.92,0.85,0.76,0.29,0.76,0.07,rabbia
7,soggetto,0.68,0.74,0.21,0.18,0.28,0.61,0.32,0.48,aspettativa
8,esterno,0.52,0.28,0.22,0.12,0.21,0.46,0.33,0.29,gioia
9,reddito,0.49,0.50,0.19,0.13,0.45,0.24,0.36,0.23,aspettativa


In [11]:
print('Score aggregato grezzo (S_e):')
print(totals.round(3).to_string())
print(f'\n=> Emozione dominante (raw): {dom_raw.upper()}')

Score aggregato grezzo (S_e):
gioia          5.031
aspettativa    7.158
rabbia         8.474
disgusto       6.684
tristezza      7.227
sorpresa       6.479
paura          7.811
fiducia        5.697

=> Emozione dominante (raw): RABBIA


## Corpus_mean normalisation (Formula 3.5 ItEm)

Lo score grezzo viene diviso per la media di corpus per ogni emozione (μ_e), ottenendo lo score normalizzato (S_e_norm). Le emozioni con valore medio alto nel corpus (come aspettativa) vengono penalizzate proporzionalmente.

In [15]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    sc_norm  = normalizza(totals.to_dict(), mu_e)
    dom_norm = emozione_dominante(sc_norm)

    df_scores = pd.DataFrame([
    {'emozione': e, 'S_e (raw)': totals[e], 'μ_e': mu_e[e], 'S_e_norm': sc_norm[e]}
    for e in BASIC_EMOTIONS
    ]).set_index('emozione').round(3)
    display(df_scores)

print(f'=> Emozione dominante (corpus_mean): {dom_norm.upper()}')

,S_e (raw),μ_e,S_e_norm
emozione,,,
gioia,5.031,6.070,0.829
aspettativa,7.158,7.226,0.991
rabbia,8.474,4.426,1.915
disgusto,6.684,3.126,2.138
tristezza,7.227,4.522,1.598
sorpresa,6.479,5.847,1.108
paura,7.811,5.006,1.560
fiducia,5.697,6.256,0.911


=> Emozione dominante (corpus_mean): DISGUSTO


In [16]:
plot_radar_single(
        sc_norm, dom_norm,
        title=f'Profilo emotivo normalizzato — commento {CID} (metodo finale)',
        height=480,
    ).show()

## Confronto: score grezzo vs corpus_mean normalizzato

In [17]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    plot_raw_vs_norm_bars(totals.to_dict(), sc_norm, CID).show()

    print(f'Lemmi usati ({len(lemmi_usati)}): {lemmi_usati}')
    print(f'Emozione dominante raw          : {dom_raw.upper()}  (score: {totals[dom_raw]:.3f})')
    print(f'Emozione dominante corpus_mean  : {dom_norm.upper()}  (score: {sc_norm[dom_norm]:.3f})')

Lemmi usati (17): ['violenza', 'bullismo', 'vietare', 'contenuto', 'costituire', 'violenza', 'bullismo', 'soggetto', 'esterno', 'reddito', 'regola', 'globale', 'violazione', 'potere', 'comportare', 'sospensione', 'sito']
Emozione dominante raw          : RABBIA  (score: 8.474)
Emozione dominante corpus_mean  : DISGUSTO  (score: 2.138)
